# z256 latent playground

The low-level tracker in this project does not receive joint targets from the
planner. It receives a **latent command**: a 256-dimensional vector `z`
produced by a *skill encoder* (DiffSR), plus two phase values. The tracker is a
frozen policy that turns `z` plus its own proprioception into 29 joint targets
at 50 Hz.

This notebook opens that interface and pokes at it:

1. encode reference motions into latents (`z` bank),
2. play a latent back through the real tracker in MuJoCo, and watch it,
3. **perturb the latent** — noise, principal directions, freezing, blending,
   rescaling — and watch what the same tracker does with it.

### What is actually loaded

Everything comes from an exported **policy bundle**: the TorchScript tracker,
the TorchScript encoder, the observation/action contracts, the normalizer, a
golden trace, and provenance (training-checkpoint SHA, encoder stride, anchor
mode). Nothing about the policy is re-implemented here, so what you see is the
deployed artifact, not a notebook approximation of it.

### Read every number with these caveats

- The plant is **MuJoCo**, not the Newton/PhysX simulator the policy trained
  in. Actuator dynamics differ, so this is a deployment-style signal, not a
  paper metric.
- Each rollout is **one deterministic episode**, no domain randomization, no
  pushes. One episode is a single sample. Treat any difference as
  **preliminary** until it repeats across seeds and starts.
- Vocabulary used below:
  - **z / latent**: the encoder's 256-dim continuous output. *Continuous*
    means unquantized — the other tracker family snaps z onto an FSQ
    lattice and has its own notebook, `fsq64_latent_perturbation.ipynb`.
  - **hold**: how many 50 Hz control ticks one z is held before the next one
    is published (10 for this bundle, so a new z every 0.2 s).
  - **renewal**: the tick where a new z is published.
  - **oracle z**: the z the encoder produces from the *reference* motion. It is
    the "correct" command; every perturbation below is defined relative to it.
  - **MPJPE-L / MPJPE-G**: mean per-joint position error against the reference,
    root-relative (L) or in the world (G), in millimetres. Only defined when a
    reference is being tracked.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from embodied_control.lowlevel.latent import (
    LatentPlayground,
    save_grid_video,
    save_video,
    video_html,
)
from embodied_control.lowlevel.publishers.latent_perturbation import (
    ConstantLatentSource,
    SequenceLatentSource,
    TransformedLatentSource,
)
from embodied_control.lowlevel.runner import verify_bundle

np.set_printoptions(precision=3, suppress=True)

## 0. Inputs

Everything the notebook needs — the policy bundles, the reference motions,
and the G1 MuJoCo model — lives in one **playkit**. If it is not already on
disk, the cell below downloads it from the private Hugging Face dataset
`GeorgiaTech/ec-latent-playkit`, **pinned to an exact revision** so what you
run is byte-for-byte what was validated. First-time setup only:

```bash
pixi run -e latent-lab python -c "from huggingface_hub import login; login()"
```

with a Hugging Face token that can read the GeorgiaTech org (or set
`HF_TOKEN`). Already have the kit? Point `EC_LATENT_PLAYKIT` at it and the
download is skipped.

### The motions

The kit ships the 30-motion `bones_seed_language30_compositionality_v1` set
(locomotion, manipulation, idle/gesture; 14,423 frames at 50 Hz). Two of the
thirty are known to be **tracker-limited** — the oracle latent itself falls
on 4 of 5 evaluation episodes in the training simulator, so a fall there
says nothing about your perturbation: `panic_run_away_180_R_001_A423` and
`walk_big_dog_ff_225_stop_R_001_A492`.

Want different motions? The full processed catalog is public — browse
[GeorgiaTech/g1_bones_seed_sonic_129k_50hz](https://huggingface.co/datasets/GeorgiaTech/g1_bones_seed_sonic_129k_50hz)
(all 129,785 clips; `g1_bones_seed_sonic_full_manifest.json` lists every
name, with language descriptions alongside) or the curated
[GeorgiaTech/g1_bones_seed_100_50hz](https://huggingface.co/datasets/GeorgiaTech/g1_bones_seed_100_50hz),
pick clip names, and ask for a kit rebuild that includes them.


In [ ]:
PLAYKIT_REPO = "GeorgiaTech/ec-latent-playkit"  # private HF dataset
PLAYKIT_REVISION = "f0cd81ed1bd7afb821ea31d148ee19aa7fab2ab1"

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PLAYKIT = Path(os.environ.get("EC_LATENT_PLAYKIT", REPO_ROOT / "assets" / "latent_playkit")).expanduser()
if not (PLAYKIT / "playkit.json").exists():
    from huggingface_hub import snapshot_download

    print(f"playkit not found at {PLAYKIT} - downloading "
          f"{PLAYKIT_REPO}@{PLAYKIT_REVISION[:12]} (~150 MB, one time)")
    snapshot_download(
        PLAYKIT_REPO,
        repo_type="dataset",
        revision=PLAYKIT_REVISION,
        local_dir=PLAYKIT,
    )

BUNDLE = PLAYKIT / "bundles" / "rollout24_gamma097_3500m"
if not BUNDLE.exists():  # v1 kits carried a single unnamed bundle
    BUNDLE = PLAYKIT / "bundle"
MODEL = PLAYKIT / "model" / "g1_29dof_rev_1_0.xml"
REFERENCE = PLAYKIT / "reference" / "root_qpos_v1"
OUTPUT = Path("runs/latent_playground")
OUTPUT.mkdir(parents=True, exist_ok=True)

missing = [str(p) for p in (BUNDLE / "manifest.json", MODEL, REFERENCE) if not p.exists()]
if missing:
    raise FileNotFoundError(
        "playkit is incomplete, missing:\n  " + "\n  ".join(missing)
        + f"\n\nSet EC_LATENT_PLAYKIT to the unpacked kit (currently {PLAYKIT})."
    )
print("playkit:", PLAYKIT.resolve())
print("bundle: ", BUNDLE.name)

In [ ]:
pg = LatentPlayground(BUNDLE, MODEL, REFERENCE)
command = pg.command
manifest = pg.bundle.manifest

print(f"interface        {manifest.interface}")
print(f"z dimensions     {pg.z_dim}  (+ {command.phase_dim} phase)")
print(f"hold             {pg.hold_steps} ticks at {pg.control_hz} Hz = "
      f"{pg.hold_steps / pg.control_hz:.2f} s per latent")
print(f"encoder input    {command.window_steps + 1} frames x {command.state_dim} "
      f"= {(command.window_steps + 1) * command.state_dim} values "
      f"({command.encoder_state_interface}, stride {command.macro_frame_stride}, "
      f"anchor {command.macro_anchor_mode})")
print(f"tracker input    {manifest.obs.total_width} values: "
      + ", ".join(f"{t.name}({t.width})" for t in manifest.obs.terms))
print(f"action           {manifest.action.width} joint targets")
print(f"checkpoint       {manifest.source['checkpoint_sha256'][:16]}...")
print(f"motions          {len(pg.motion_names)}")
for name in pg.motion_names:
    print("   ", name)

### Provenance gate

`verify_bundle` replays the exporter's golden trace through the shipped
TorchScript. If this passes, the policy and encoder in this notebook produce
bit-comparable outputs to the training checkpoint they were exported from. If
it fails, stop — nothing below means anything.

In [ ]:
report = verify_bundle(BUNDLE)
print(f"policy rows  {report['policy_rows']}, max abs err {report['policy_max_abs_err']:.2e}")
print(f"encoder rows {report.get('encoder_rows')}, max abs err "
      f"{report.get('encoder_max_abs_err', float('nan')):.2e}")

## 1. The latent bank

`encode_motion` runs the encoder over a motion at the bundle's hold spacing:
one latent every 10 frames, each from a 10-frame lookahead window. That is
exactly the sequence of latents a perfectly tracking rollout would consume.

Encoding is offline here, so the expert anchor is re-expressed in the
*reference's own* anchor frame — the perfect-tracking assumption. In a closed
loop the encoder sees the real robot anchor instead, so a drifting robot gets a
slightly different oracle z. That difference is itself worth an experiment.

In [ ]:
banks = {name: pg.encode_motion(name) for name in pg.motion_names}
Z_ALL = np.concatenate([bank.z for bank in banks.values()])
print(f"{len(banks)} motions -> {Z_ALL.shape[0]} latents of width {Z_ALL.shape[1]}")

pd.DataFrame(
    [
        {
            "motion": name,
            "latents": len(bank),
            "frames": int(bank.cursors[-1] + 1),
            "|z| mean": round(float(np.linalg.norm(bank.z, axis=1).mean()), 2),
            "z min": round(float(bank.z.min()), 2),
            "z max": round(float(bank.z.max()), 2),
        }
        for name, bank in banks.items()
    ]
)

In [ ]:
Z_MEAN = Z_ALL.mean(axis=0)
Z_STD = Z_ALL.std(axis=0)
centered = Z_ALL - Z_MEAN
_, singular, components = np.linalg.svd(centered, full_matrices=False)
variance = singular ** 2 / (centered.shape[0] - 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 3.4))
axes[0].hist(Z_ALL.reshape(-1), bins=80, color="#4c72b0")
axes[0].set_title("all latent values")
axes[1].plot(np.sort(Z_STD)[::-1], color="#c44e52")
axes[1].set_title("per-dimension std (sorted)")
axes[1].set_xlabel("dimension rank")
axes[2].plot(np.cumsum(variance) / variance.sum(), marker=".", color="#55a868")
axes[2].set_title("PCA cumulative variance")
axes[2].set_xlabel("component")
axes[2].set_xlim(0, 40)
axes[2].grid(alpha=0.3)
plt.tight_layout()
print(f"components for 90% of the variance: {int(np.searchsorted(np.cumsum(variance) / variance.sum(), 0.9)) + 1}")
print(f"dead dimensions (std < 1e-3): {int((Z_STD < 1e-3).sum())}")

In [ ]:
# Where do motions sit relative to each other? Cosine similarity of mean z.
names = list(banks)
means = np.stack([banks[name].z.mean(axis=0) for name in names])
unit = means / np.linalg.norm(means, axis=1, keepdims=True)
similarity = unit @ unit.T

fig, ax = plt.subplots(figsize=(7, 6))
image = ax.imshow(similarity, cmap="viridis")
ax.set_xticks(range(len(names)), [n[:22] for n in names], rotation=90)
ax.set_yticks(range(len(names)), [n[:22] for n in names])
fig.colorbar(image, label="cosine similarity of mean z")
plt.tight_layout()

In [ ]:
# The latent trajectory of one motion: 20 highest-variance dimensions over time.
DEMO = "walk_arc_cw_start_R_slow_001_A443"
demo_motion = pg.motion(DEMO)
demo_bank = banks[DEMO]
top_dims = np.argsort(demo_bank.z.std(axis=0))[::-1][:20]

fig, ax = plt.subplots(figsize=(11, 3.2))
image = ax.imshow(demo_bank.z[:, top_dims].T, aspect="auto", cmap="coolwarm")
ax.set_xlabel("renewal (one per 0.2 s)")
ax.set_ylabel("top-variance dimension")
ax.set_title(f"z over time - {DEMO}")
fig.colorbar(image)
plt.tight_layout()

## 2. Baseline: the oracle latent

`make_reference_source` is the certified path — reference window in, encoder,
z out, held for 10 ticks — and it is bit-identical to what the deployment
runtime publishes (asserted in `tests/lowlevel/test_latent_playground.py`).
Everything after this is measured against this row.

In [ ]:
STEPS = 300  # 6 seconds at 50 Hz

baseline = pg.rollout(
    pg.make_reference_source(demo_motion),
    motion=demo_motion,
    max_steps=STEPS,
)
print(pg.summary(baseline, demo_motion))
video_html(save_video(baseline, OUTPUT / "baseline.mp4"))

## 3. Noise on the latent

The first question an encoder invites: how much does the latent have to move
before the behaviour breaks? Here each renewal's oracle z gets Gaussian noise
scaled by the *per-dimension* std of the bank, so `sigma = 0.5` means "half a
latent-population standard deviation of noise on every dimension, resampled
every 0.2 s".

Watch two different failure modes: the motion first becomes sloppy (MPJPE
rises while the robot stays up), then it falls.

In [ ]:
def noise_transform(sigma, seed=0):
    """Add sigma * per-dimension-std Gaussian noise at every renewal."""
    rng = np.random.default_rng(seed)
    scale = float(sigma) * Z_STD

    def transform(z, renewal_index):
        return z + scale * rng.standard_normal(z.shape).astype(np.float32)

    return transform


SIGMAS = [0.0, 0.25, 0.5, 1.0, 2.0]
noise_rollouts = {}
rows = []
for sigma in SIGMAS:
    source = TransformedLatentSource(
        pg.make_reference_source(demo_motion), noise_transform(sigma)
    )
    roll = pg.rollout(source, motion=demo_motion, max_steps=STEPS)
    label = f"sigma={sigma}"
    noise_rollouts[label] = roll
    rows.append({"sigma": sigma, **pg.summary(roll, demo_motion)})

noise_table = pd.DataFrame(rows).drop(columns=["motion"])
noise_table

In [ ]:
video_html(save_grid_video(noise_rollouts, OUTPUT / "noise_sweep.mp4", columns=3), width=900)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
for label, roll in noise_rollouts.items():
    axes[0].plot(roll.base_height, label=label)
axes[0].axhline(0.4, color="k", ls="--", lw=1, label="fall threshold")
axes[0].set_xlabel("control tick")
axes[0].set_ylabel("pelvis height (m)")
axes[0].legend(fontsize=8)
if "mpjpe_l_mm" in noise_table:
    axes[1].plot(noise_table["sigma"], noise_table["mpjpe_l_mm"], marker="o")
    axes[1].set_ylabel("MPJPE-L (mm)")
axes[1].set_xlabel("noise sigma (population std)")
axes[1].set_title("tracking error vs latent noise")
axes[1].grid(alpha=0.3)
plt.tight_layout()
print("survived:", dict(zip(noise_table["sigma"], noise_table["survived"])))

## 4. Moving along principal directions

Isotropic noise treats all 256 dimensions alike, but the bank is far from
isotropic. This experiment pushes the oracle z along one principal component
at a time, by a multiple of that component's own standard deviation, and keeps
everything else untouched. A component that changes the motion in a *readable*
way (stride length, arm height, turn rate) is a candidate interpretable
direction; most will not be.

In [ ]:
def direction_transform(direction, magnitude):
    """Shift every latent by magnitude * direction (unit vector)."""
    shift = (float(magnitude) * np.asarray(direction, dtype=np.float32)).astype(np.float32)

    def transform(z, renewal_index):
        return z + shift

    return transform


COMPONENTS = [0, 1, 2]
AMPLITUDES = [-2.0, 2.0]
direction_rollouts = {}
rows = []
for component in COMPONENTS:
    sigma_c = float(np.sqrt(variance[component]))
    for amplitude in AMPLITUDES:
        source = TransformedLatentSource(
            pg.make_reference_source(demo_motion),
            direction_transform(components[component], amplitude * sigma_c),
        )
        roll = pg.rollout(source, motion=demo_motion, max_steps=STEPS)
        label = f"PC{component} {amplitude:+.0f} sd"
        direction_rollouts[label] = roll
        rows.append({"component": component, "amplitude_sd": amplitude,
                     **pg.summary(roll, demo_motion)})

pd.DataFrame(rows).drop(columns=["motion"])

In [ ]:
video_html(save_grid_video(direction_rollouts, OUTPUT / "pca_directions.mp4", columns=2), width=760)

## 5. Freezing one latent

Drop the reference entirely: publish **one fixed z** forever, starting from the
pose that z was encoded at. The tracker now has a constant command with a
running phase, and whatever it settles into is that latent's *attractor* —
often a looping or standing behaviour rather than the original clip.

This is the cleanest "what does this latent mean" probe, because nothing else
is changing.

In [ ]:
FREEZE_AT = [0, len(demo_bank) // 3, 2 * len(demo_bank) // 3]
freeze_rollouts = {}
rows = []
for index in FREEZE_AT:
    cursor = int(demo_bank.cursors[index])
    roll = pg.rollout(
        ConstantLatentSource(demo_bank.z[index]),
        motion=demo_motion,
        start_pose_frame=cursor,
        max_steps=STEPS,
    )
    label = f"frozen @frame {cursor}"
    freeze_rollouts[label] = roll
    rows.append({"encoded_at_frame": cursor, **pg.summary(roll)})

pd.DataFrame(rows)

In [ ]:
video_html(save_grid_video(freeze_rollouts, OUTPUT / "frozen_latents.mp4", columns=3), width=900)

In [ ]:
# Same probe across motions: freeze each motion's mid-clip latent.
cross_rollouts = {}
rows = []
for name, bank in banks.items():
    index = len(bank) // 2
    cursor = int(bank.cursors[index])
    motion = pg.motion(name)
    roll = pg.rollout(
        ConstantLatentSource(bank.z[index]),
        motion=motion,
        start_pose_frame=cursor,
        max_steps=200,
    )
    cross_rollouts[name[:20]] = roll
    rows.append({**pg.summary(roll), "motion": name[:28], "frozen_at_frame": cursor})

pd.DataFrame(rows)

In [ ]:
video_html(save_grid_video(cross_rollouts, OUTPUT / "frozen_across_motions.mp4", columns=5), width=1100)

## 6. Blending two latents

If the latent space is smooth, a convex blend of two motions' latents should
produce something between the two behaviours; if it is not, the blend falls
off a cliff somewhere in the middle. Both outcomes are informative — the second
says the planner must not interpolate naively between skills.

In [ ]:
MOTION_A = "walk_arc_cw_start_R_slow_001_A443"
MOTION_B = "casual_greeting_R_001_A428"
ALPHAS = [0.0, 0.25, 0.5, 0.75, 1.0]

z_a = banks[MOTION_A].z[len(banks[MOTION_A]) // 2]
z_b = banks[MOTION_B].z[len(banks[MOTION_B]) // 2]
start_motion = pg.motion(MOTION_A)
start_frame = int(banks[MOTION_A].cursors[len(banks[MOTION_A]) // 2])

blend_rollouts = {}
rows = []
for alpha in ALPHAS:
    z = ((1.0 - alpha) * z_a + alpha * z_b).astype(np.float32)
    roll = pg.rollout(
        ConstantLatentSource(z),
        motion=start_motion,
        start_pose_frame=start_frame,
        max_steps=200,
    )
    blend_rollouts[f"alpha={alpha}"] = roll
    rows.append({"alpha": alpha, "|z|": round(float(np.linalg.norm(z)), 2), **pg.summary(roll)})

pd.DataFrame(rows).drop(columns=["motion"])

In [ ]:
video_html(save_grid_video(blend_rollouts, OUTPUT / "blend.mp4", columns=5), width=1100)

## 7. Rescaling the latent

Scaling z is not a semantic operation — it moves the command off the manifold
the encoder ever produced. That is the point: it measures how much of the
tracker's behaviour is driven by the *direction* of z versus its magnitude, and
how gracefully the policy degrades when a planner emits an out-of-distribution
latent (which a regression head will do).

In [ ]:
GAINS = [0.0, 0.5, 1.0, 1.5]
gain_rollouts = {}
rows = []
for gain in GAINS:
    source = TransformedLatentSource(
        pg.make_reference_source(demo_motion),
        lambda z, index, gain=gain: (gain * z).astype(np.float32),
    )
    roll = pg.rollout(source, motion=demo_motion, max_steps=STEPS)
    gain_rollouts[f"z x {gain}"] = roll
    rows.append({"gain": gain, **pg.summary(roll, demo_motion)})

pd.DataFrame(rows).drop(columns=["motion"])

In [ ]:
video_html(save_grid_video(gain_rollouts, OUTPUT / "gain.mp4", columns=4), width=1000)

## 8. Your turn

`TransformedLatentSource(inner, fn)` accepts any `fn(z, renewal_index) -> z`,
so a new experiment is one function. Some that are worth running next:

- **single-dimension sweep**: zero or sweep one dimension at a time to find
  dimensions the tracker ignores (the bank already shows how many are near
  dead);
- **stale latent**: republish the z from N renewals ago to measure how much
  latency the interface tolerates;
- **cross-motion swap**: start from motion A's pose and feed motion B's latent
  sequence with `SequenceLatentSource`;
- **encoder inversion**: optimise a z so the rollout matches a target pose —
  the playground gives you the forward map, and it is cheap (a 300-step episode
  is ~1 s on CPU).

Anything you find here is a hypothesis about the interface. Before it becomes a
claim, repeat it across seeds and starts, and confirm it in the training
simulator — MuJoCo actuator dynamics are not the ones the policy learned in.

In [ ]:
def my_transform(z, renewal_index):
    """Edit me: return a modified copy of the oracle latent."""
    out = z.copy()
    out[:8] = 0.0  # example: silence the eight lowest-index dimensions
    return out


custom = pg.rollout(
    TransformedLatentSource(pg.make_reference_source(demo_motion), my_transform),
    motion=demo_motion,
    max_steps=STEPS,
)
print(pg.summary(custom, demo_motion))
video_html(save_video(custom, OUTPUT / "custom.mp4"))